# Oiled — Stage 2: SAR Segmentation Baseline
**CNN Triage Classifier + 2-Channel U-Net (ResNet-34)**  
Input: Sentinel-1 VV/VH Sigma0 dB · Output: Oil probability mask + triage label

In [ ]:
# ─── CELL 0: GPU VERIFICATION ───────────────────────────────────────────────
import torch
import platform

print('=== Oiled — Stage 2: SAR Model Training ===')
print(f'Python:  {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')
print(f'GPUs:    {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  VRAM={p.total_memory//1024**2} MiB')

if not torch.cuda.is_available():
    raise RuntimeError('GPU not available — enable GPU in notebook settings.')
print('\nEnvironment check passed.')

In [ ]:
# ─── CELL 1: INSTALL DEPENDENCIES ───────────────────────────────────────────
# FIX: smp 0.3.4 requires timm==0.9.7 exactly; install compatible versions first.
import subprocess, sys

def pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *args], check=True)

pip('timm==0.9.7')                          # must be installed BEFORE smp
pip('segmentation-models-pytorch==0.3.4')
pip('rasterio==1.4.3')
pip('albumentations==1.4.23')
pip('scikit-learn')

print('All dependencies installed.')

In [ ]:
# ─── CELL 2: AUTO-DISCOVER DATASET PATHS ────────────────────────────────────
# FIX: Do NOT hardcode paths. Scan /kaggle/input/ at runtime to find actual mounts.
from pathlib import Path

INPUT = Path('/kaggle/input')
OUT   = Path('/kaggle/working')

print('=== /kaggle/input directory structure ===')
for d in sorted(INPUT.iterdir()):
    if d.is_dir():
        tif_count = len(list(d.rglob('*.tif')))
        sub = sorted([s.relative_to(INPUT) for s in d.rglob('*') if s.is_dir()])[:4]
        print(f'  {d.name}  ({tif_count} .tif files)  subdirs: {sub}')

def first_tif_parent(base: Path) -> Path | None:
    """Return the deepest directory that directly contains .tif files."""
    for p in sorted(base.rglob('*.tif')):
        return p.parent
    return None

def match_dataset(keyword: str) -> Path | None:
    """Find the dataset directory whose name contains keyword, then find its tif parent."""
    for d in sorted(INPUT.iterdir()):
        if keyword in d.name.lower():
            parent = first_tif_parent(d)
            if parent:
                return parent
    return None

# Discover paths dynamically
OIL_IMG_DIR  = match_dataset('oil-spill-train')
LIKE_IMG_DIR = match_dataset('lookalike')
NOIL_IMG_DIR = match_dataset('no-oil')

# For masks: find the directory that contains mask-style tifs (much smaller files ~4 MB)
def find_mask_dir() -> Path | None:
    for d in sorted(INPUT.iterdir()):
        if 'dartis' in d.name.lower() or 'mask' in d.name.lower():
            for p in sorted(d.rglob('*.tif')):
                return p.parent
    return None

OIL_MASK_DIR = find_mask_dir()

print('\n=== Resolved dataset paths ===')
for name, path in [('Oil images', OIL_IMG_DIR), ('Oil masks', OIL_MASK_DIR),
                   ('Lookalike',  LIKE_IMG_DIR), ('No-oil',   NOIL_IMG_DIR)]:
    count = len(list(path.glob('*.tif'))) if path else 0
    print(f'  {name:14s}: {path}  ({count} files)')

if OIL_IMG_DIR is None or OIL_MASK_DIR is None:
    raise RuntimeError(
        'Could not find oil images or masks.\n'
        'Check that all four datasets are attached in kernel-metadata.json.\n'
        'Attached datasets: ' + str([d.name for d in INPUT.iterdir() if d.is_dir()])
    )

In [ ]:
# ─── CELL 3: CONFIGURATION ──────────────────────────────────────────────────
import json, torch

N_GPU = torch.cuda.device_count()
TILE  = 512

CFG = {
    # Data
    'tile_size':          TILE,
    'tile_stride':        384,     # 128 px overlap
    'num_oil_scenes':     200,     # starter subset per class
    'val_fraction':       0.15,
    'seed':               42,
    'max_tiles_per_scene': 32,
    # U-Net
    'seg_encoder':        'resnet34',
    'in_channels':        2,       # VV + VH
    'batch_size':         16,      # per GPU → 32 effective on 2x T4
    'seg_epochs':         40,
    'seg_lr':             1e-4,
    'seg_weight_decay':   1e-4,
    'seg_patience':       8,
    'bce_weight':         0.5,
    # CNN Triage
    'triage_classes':     3,       # oil=0, lookalike=1, clean=2
    'triage_tile':        256,
    'triage_batch':       32,      # per GPU
    'triage_epochs':      30,
    'triage_lr':          1e-4,
    'triage_weight_decay': 1e-4,
    'triage_patience':    6,
    # Export
    'onnx_opset':         17,
}
print(json.dumps(CFG, indent=2))
print(f'\nEffective seg batch size  : {CFG["batch_size"] * N_GPU}')
print(f'Effective triage batch    : {CFG["triage_batch"] * N_GPU}')

In [ ]:
# ─── CELL 4: DATASET DISCOVERY & MANIFEST ───────────────────────────────────
import random
import numpy as np
import pandas as pd

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])

def scan_paired(img_dir: Path, mask_dir: Path, label: str) -> list:
    records = []
    if img_dir is None or mask_dir is None:
        return records
    for img_path in sorted(img_dir.glob('*.tif')):
        mask_path = mask_dir / img_path.name
        if mask_path.exists():
            records.append({'image': str(img_path), 'mask': str(mask_path),
                            'label': label, 'stem': img_path.stem})
    return records

def scan_images(img_dir: Path, label: str) -> list:
    records = []
    if img_dir is None:
        return records
    for p in sorted(img_dir.rglob('*.tif')):
        records.append({'image': str(p), 'mask': None,
                        'label': label, 'stem': p.stem})
    return records

oil_pairs  = scan_paired(OIL_IMG_DIR, OIL_MASK_DIR, 'oil')
like_imgs  = scan_images(LIKE_IMG_DIR, 'lookalike')
noil_imgs  = scan_images(NOIL_IMG_DIR, 'clean')

print(f'Oil scenes with paired masks : {len(oil_pairs)}')
print(f'Lookalike scenes             : {len(like_imgs)}')
print(f'No-oil (clean) scenes        : {len(noil_imgs)}')

if len(oil_pairs) == 0:
    raise RuntimeError(
        'Zero paired oil+mask scenes found.\n'
        f'OIL_IMG_DIR  = {OIL_IMG_DIR}\n'
        f'OIL_MASK_DIR = {OIL_MASK_DIR}\n'
        'Check that file names match between image and mask directories.'
    )

n = CFG['num_oil_scenes']
oil_sample  = random.sample(oil_pairs, min(n, len(oil_pairs)))
like_sample = random.sample(like_imgs, min(n, len(like_imgs)))
noil_sample = random.sample(noil_imgs, min(n, len(noil_imgs)))

all_records = oil_sample + like_sample + noil_sample
manifest = pd.DataFrame(all_records)
manifest = manifest.sample(frac=1, random_state=CFG['seed']).reset_index(drop=True)
manifest.to_csv(OUT / 'manifest.csv', index=False)

print(f'\nManifest size : {len(manifest)} scenes')
print(manifest['label'].value_counts().to_string())

In [ ]:
# ─── CELL 5: SCENE-LEVEL TRAIN / VAL SPLIT ──────────────────────────────────
from sklearn.model_selection import train_test_split

def split_df(df, val_frac, seed):
    tr, va = train_test_split(df, test_size=val_frac, random_state=seed)
    return tr.reset_index(drop=True), va.reset_index(drop=True)

VF, SD = CFG['val_fraction'], CFG['seed']

oil_df  = manifest[manifest.label == 'oil'].copy()
like_df = manifest[manifest.label == 'lookalike'].copy()
noil_df = manifest[manifest.label == 'clean'].copy()

oil_tr,  oil_va  = split_df(oil_df,  VF, SD)
like_tr, like_va = split_df(like_df, VF, SD) if len(like_df) >= 2 else (like_df, like_df)
noil_tr, noil_va = split_df(noil_df, VF, SD) if len(noil_df) >= 2 else (noil_df, noil_df)

seg_train_df = oil_tr
seg_val_df   = oil_va

tri_train_df = pd.concat([oil_tr, like_tr, noil_tr]).sample(frac=1, random_state=SD).reset_index(drop=True)
tri_val_df   = pd.concat([oil_va, like_va, noil_va]).sample(frac=1, random_state=SD).reset_index(drop=True)

print('Segmentation (oil only):')
print(f'  train={len(seg_train_df)}  val={len(seg_val_df)}')
print('Triage (all classes):')
print(f'  train={tri_train_df.label.value_counts().to_dict()}')
print(f'  val  ={tri_val_df.label.value_counts().to_dict()}')

In [ ]:
# ─── CELL 6: SAR I/O & AUGMENTATION ─────────────────────────────────────────
import rasterio
from rasterio.errors import NotGeoreferencedWarning
import albumentations as A
from albumentations.pytorch import ToTensorV2
import warnings
warnings.filterwarnings('ignore', category=NotGeoreferencedWarning)

SAR_MIN, SAR_MAX = -40.0, 10.0
TRIAGE_TILE = CFG['triage_tile']

def read_sar(path: str) -> np.ndarray:
    """Read SAR scene → (H, W, 2) float32 normalised to [0,1]."""
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)   # (C, H, W)
    if arr.shape[0] == 1:                     # VV-only → duplicate to 2ch
        arr = np.concatenate([arr, arr], axis=0)
    arr = arr[:2]
    arr = np.clip(arr, SAR_MIN, SAR_MAX)
    arr = (arr - SAR_MIN) / (SAR_MAX - SAR_MIN)
    return np.transpose(arr, (1, 2, 0))       # (H, W, 2)

def read_mask(path: str) -> np.ndarray:
    with rasterio.open(path) as src:
        m = src.read(1).astype(np.uint8)
    return np.clip(m, 0, 1)

def extract_tiles(img, mask, tile=TILE, stride=CFG['tile_stride']):
    H, W = img.shape[:2]
    for y in range(0, max(H - tile + 1, 1), stride):
        for x in range(0, max(W - tile + 1, 1), stride):
            it = img[y:y+tile, x:x+tile]
            mt = mask[y:y+tile, x:x+tile] if mask is not None else np.zeros((tile, tile), np.uint8)
            if it.shape[:2] == (tile, tile):
                yield it, mt

SEG_TRAIN_AUG = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.GaussNoise(var_limit=(0.002, 0.015), p=0.4),
    A.RandomBrightnessContrast(0.1, 0.1, p=0.3),
    ToTensorV2(),
])
SEG_VAL_AUG = A.Compose([ToTensorV2()])

TRIAGE_TRAIN_AUG = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.GaussNoise(var_limit=(0.002, 0.015), p=0.4),
    A.CenterCrop(TRIAGE_TILE, TRIAGE_TILE),
    ToTensorV2(),
])
TRIAGE_VAL_AUG = A.Compose([A.CenterCrop(TRIAGE_TILE, TRIAGE_TILE), ToTensorV2()])

print('SAR I/O and augmentation pipelines ready.')

In [ ]:
# ─── CELL 7: PYTORCH DATASETS ────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

MAX_TILES = CFG['max_tiles_per_scene']

class SegmentationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, augment: bool):
        self._aug = SEG_TRAIN_AUG if augment else SEG_VAL_AUG
        self.tiles = []
        skipped = 0
        for _, row in df.iterrows():
            try:
                img  = read_sar(row['image'])
                mask = read_mask(row['mask'])
                count = 0
                for it, mt in extract_tiles(img, mask):
                    self.tiles.append((it.copy(), mt.copy()))
                    count += 1
                    if count >= MAX_TILES:
                        break
            except Exception as e:
                skipped += 1
                if skipped <= 3:
                    print(f'  WARN skip {Path(row["image"]).name}: {e}')
        print(f'  [{"train" if augment else "val"}] {len(self.tiles)} tiles '
              f'from {len(df)-skipped}/{len(df)} scenes')

    def __len__(self): return len(self.tiles)

    def __getitem__(self, idx):
        img, mask = self.tiles[idx]
        out = self._aug(image=img, mask=mask)
        return out['image'].float(), out['mask'].float().unsqueeze(0)


TRIAGE_LABELS = {'oil': 0, 'lookalike': 1, 'clean': 2}

class TriageDataset(Dataset):
    def __init__(self, df: pd.DataFrame, augment: bool):
        self._aug = TRIAGE_TRAIN_AUG if augment else TRIAGE_VAL_AUG
        self.items = []
        skipped = 0
        for _, row in df.iterrows():
            label_idx = TRIAGE_LABELS.get(row['label'], 2)
            try:
                img = read_sar(row['image'])
                H, W = img.shape[:2]
                if H < TRIAGE_TILE or W < TRIAGE_TILE:
                    ph = max(0, TRIAGE_TILE - H)
                    pw = max(0, TRIAGE_TILE - W)
                    img = np.pad(img, ((0, ph), (0, pw), (0, 0)), mode='reflect')
                out = self._aug(image=img)
                self.items.append((out['image'].float(), label_idx))
            except Exception as e:
                skipped += 1
                if skipped <= 3:
                    print(f'  WARN skip {Path(row["image"]).name}: {e}')
        print(f'  [{"train" if augment else "val"}] {len(self.items)} triage items')

    def __len__(self): return len(self.items)
    def __getitem__(self, idx): return self.items[idx]

print('Dataset classes defined.')

In [ ]:
# ─── CELL 8: BUILD DATALOADERS ───────────────────────────────────────────────
WORKERS = 4

print('Building segmentation datasets...')
seg_train_ds = SegmentationDataset(seg_train_df, augment=True)
seg_val_ds   = SegmentationDataset(seg_val_df,   augment=False)

seg_train_dl = DataLoader(seg_train_ds, batch_size=CFG['batch_size']*N_GPU,
                          shuffle=True,  num_workers=WORKERS, pin_memory=True, drop_last=True)
seg_val_dl   = DataLoader(seg_val_ds,   batch_size=CFG['batch_size']*N_GPU,
                          shuffle=False, num_workers=WORKERS, pin_memory=True)

print('\nBuilding triage datasets...')
tri_train_ds = TriageDataset(tri_train_df, augment=True)
tri_val_ds   = TriageDataset(tri_val_df,   augment=False)

tri_train_dl = DataLoader(tri_train_ds, batch_size=CFG['triage_batch']*N_GPU,
                          shuffle=True,  num_workers=WORKERS, pin_memory=True, drop_last=True)
tri_val_dl   = DataLoader(tri_val_ds,   batch_size=CFG['triage_batch']*N_GPU,
                          shuffle=False, num_workers=WORKERS, pin_memory=True)

print(f'\nSeg   batches: train={len(seg_train_dl)}  val={len(seg_val_dl)}')
print(f'Triage batches: train={len(tri_train_dl)}  val={len(tri_val_dl)}')

In [ ]:
# ─── CELL 9: MODEL DEFINITIONS ───────────────────────────────────────────────
import segmentation_models_pytorch as smp
import torch.nn as nn
import timm

DEVICE = torch.device('cuda')

# ── U-Net ──────────────────────────────────────────────────────────────────────
seg_model = smp.Unet(
    encoder_name    = CFG['seg_encoder'],
    encoder_weights = None,
    in_channels     = CFG['in_channels'],
    classes         = 1,
    activation      = None,          # raw logits; sigmoid in loss/metrics
)
if N_GPU > 1:
    seg_model = nn.DataParallel(seg_model)
    print(f'U-Net → DataParallel across {N_GPU} GPUs.')
seg_model = seg_model.to(DEVICE)
print(f'U-Net params: {sum(p.numel() for p in seg_model.parameters())/1e6:.1f}M')

# ── Triage CNN ─────────────────────────────────────────────────────────────────
class TriageCNN(nn.Module):
    def __init__(self, num_classes=3, in_channels=2):
        super().__init__()
        self.backbone = timm.create_model(
            'resnet18', pretrained=False,
            num_classes=num_classes, in_chans=in_channels,
        )
    def forward(self, x): return self.backbone(x)

tri_model = TriageCNN(num_classes=CFG['triage_classes'], in_channels=CFG['in_channels'])
if N_GPU > 1:
    tri_model = nn.DataParallel(tri_model)
tri_model = tri_model.to(DEVICE)
print(f'Triage CNN params: {sum(p.numel() for p in tri_model.parameters())/1e6:.1f}M')

In [ ]:
# ─── CELL 10: LOSS FUNCTIONS & METRICS ──────────────────────────────────────

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, smooth=1.0):
        super().__init__()
        self.w = bce_weight
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    def dice(self, logits, t):
        p = torch.sigmoid(logits)
        num = 2.0*(p*t).sum(dim=(2,3)) + self.smooth
        den = p.sum(dim=(2,3)) + t.sum(dim=(2,3)) + self.smooth
        return 1.0 - (num/den).mean()
    def forward(self, logits, t):
        return self.w*self.bce(logits,t) + (1-self.w)*self.dice(logits,t)

def iou_score(logits, t, thresh=0.5, s=1.0):
    p = (torch.sigmoid(logits) > thresh).float()
    inter = (p*t).sum(); union = p.sum()+t.sum()-inter
    return ((inter+s)/(union+s)).item()

def dice_score(logits, t, thresh=0.5, s=1.0):
    p = (torch.sigmoid(logits) > thresh).float()
    return ((2.0*(p*t).sum()+s)/(p.sum()+t.sum()+s)).item()

def accuracy(logits, labels):
    return (logits.argmax(1) == labels).float().mean().item()

seg_criterion = BCEDiceLoss(bce_weight=CFG['bce_weight'])
tri_criterion = nn.CrossEntropyLoss()
print('Losses and metrics defined.')

In [ ]:
# ─── CELL 11: U-NET TRAINING ─────────────────────────────────────────────────
from torch.cuda.amp import GradScaler, autocast
import time

def train_segmentation():
    opt = torch.optim.AdamW(seg_model.parameters(),
                            lr=CFG['seg_lr'], weight_decay=CFG['seg_weight_decay'])
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['seg_epochs'])
    scaler = GradScaler()
    best_iou, pat, history = 0.0, 0, []

    for ep in range(1, CFG['seg_epochs']+1):
        t0 = time.time()
        # Train
        seg_model.train(); tr_loss = 0.0
        for xb, yb in seg_train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            with autocast():
                loss = seg_criterion(seg_model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tr_loss += loss.item()
        tr_loss /= len(seg_train_dl)
        # Val
        seg_model.eval(); va_loss = va_iou = va_dice = 0.0
        with torch.no_grad():
            for xb, yb in seg_val_dl:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                with autocast():
                    logits = seg_model(xb)
                    va_loss += seg_criterion(logits, yb).item()
                va_iou  += iou_score(logits, yb)
                va_dice += dice_score(logits, yb)
        va_loss /= len(seg_val_dl); va_iou /= len(seg_val_dl); va_dice /= len(seg_val_dl)
        sched.step(); lr = sched.get_last_lr()[0]
        history.append(dict(epoch=ep, tr_loss=tr_loss, va_loss=va_loss,
                            va_iou=va_iou, va_dice=va_dice, lr=lr))
        print(f'Ep {ep:3d}/{CFG["seg_epochs"]} | '
              f'tr={tr_loss:.4f}  va={va_loss:.4f}  '
              f'IoU={va_iou:.4f}  Dice={va_dice:.4f}  lr={lr:.2e}  {time.time()-t0:.0f}s')
        if va_iou > best_iou:
            best_iou = va_iou; pat = 0
            state = seg_model.module.state_dict() if isinstance(seg_model, nn.DataParallel) \
                    else seg_model.state_dict()
            torch.save({'epoch': ep, 'model_state': state,
                        'val_iou': va_iou, 'val_dice': va_dice, 'config': CFG},
                       OUT/'seg_best.pt')
            print(f'  ✓ checkpoint (IoU={best_iou:.4f})')
        else:
            pat += 1
            if pat >= CFG['seg_patience']:
                print(f'  Early stop @ ep {ep}'); break
    return history, best_iou

print('─── U-Net Training ───')
seg_history, best_seg_iou = train_segmentation()
print(f'\n✓ Best Val IoU = {best_seg_iou:.4f}')

In [ ]:
# ─── CELL 12: CNN TRIAGE TRAINING ────────────────────────────────────────────

def train_triage():
    opt = torch.optim.AdamW(tri_model.parameters(),
                            lr=CFG['triage_lr'], weight_decay=CFG['triage_weight_decay'])
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['triage_epochs'])
    scaler = GradScaler()
    best_acc, pat, history = 0.0, 0, []

    for ep in range(1, CFG['triage_epochs']+1):
        t0 = time.time()
        # Train
        tri_model.train(); tr_loss = tr_acc = 0.0
        for xb, yb in tri_train_dl:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE) if isinstance(yb, torch.Tensor) else torch.tensor(yb).to(DEVICE)
            opt.zero_grad()
            with autocast():
                logits = tri_model(xb); loss = tri_criterion(logits, yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tr_loss += loss.item(); tr_acc += accuracy(logits, yb)
        tr_loss /= len(tri_train_dl); tr_acc /= len(tri_train_dl)
        # Val
        tri_model.eval(); va_loss = va_acc = 0.0
        with torch.no_grad():
            for xb, yb in tri_val_dl:
                xb = xb.to(DEVICE)
                yb = yb.to(DEVICE) if isinstance(yb, torch.Tensor) else torch.tensor(yb).to(DEVICE)
                with autocast():
                    logits = tri_model(xb); loss = tri_criterion(logits, yb)
                va_loss += loss.item(); va_acc += accuracy(logits, yb)
        va_loss /= len(tri_val_dl); va_acc /= len(tri_val_dl)
        sched.step(); lr = sched.get_last_lr()[0]
        history.append(dict(epoch=ep, tr_loss=tr_loss, tr_acc=tr_acc,
                            va_loss=va_loss, va_acc=va_acc, lr=lr))
        print(f'Ep {ep:3d}/{CFG["triage_epochs"]} | '
              f'tr={tr_loss:.4f} acc={tr_acc:.3f} | '
              f'va={va_loss:.4f} acc={va_acc:.3f} | lr={lr:.2e}  {time.time()-t0:.0f}s')
        if va_acc > best_acc:
            best_acc = va_acc; pat = 0
            state = tri_model.module.state_dict() if isinstance(tri_model, nn.DataParallel) \
                    else tri_model.state_dict()
            torch.save({'epoch': ep, 'model_state': state,
                        'val_acc': va_acc, 'config': CFG, 'label_map': TRIAGE_LABELS},
                       OUT/'triage_best.pt')
            print(f'  ✓ checkpoint (acc={best_acc:.3f})')
        else:
            pat += 1
            if pat >= CFG['triage_patience']:
                print(f'  Early stop @ ep {ep}'); break
    return history, best_acc

print('─── Triage CNN Training ───')
tri_history, best_tri_acc = train_triage()
print(f'\n✓ Best Val Acc = {best_tri_acc:.4f}')

In [ ]:
# ─── CELL 13: ONNX EXPORT ────────────────────────────────────────────────────

def export_unet_onnx():
    m = smp.Unet(encoder_name=CFG['seg_encoder'], encoder_weights=None,
                 in_channels=CFG['in_channels'], classes=1, activation='sigmoid').to(DEVICE).eval()
    ckpt = torch.load(OUT/'seg_best.pt', map_location=DEVICE)
    m.load_state_dict(ckpt['model_state'])
    dummy = torch.randn(1, 2, TILE, TILE, device=DEVICE)
    out_p = OUT/'unet_segmentation.onnx'
    torch.onnx.export(m, dummy, str(out_p), opset_version=CFG['onnx_opset'],
                      input_names=['sar_tile'], output_names=['oil_probability'],
                      dynamic_axes={'sar_tile': {0:'batch'}, 'oil_probability': {0:'batch'}})
    print(f'✓ U-Net ONNX: {out_p.name}  ({out_p.stat().st_size/1e6:.1f} MB)')

def export_triage_onnx():
    m = TriageCNN(num_classes=CFG['triage_classes'], in_channels=CFG['in_channels']).to(DEVICE).eval()
    ckpt = torch.load(OUT/'triage_best.pt', map_location=DEVICE)
    m.load_state_dict(ckpt['model_state'])
    dummy = torch.randn(1, 2, TRIAGE_TILE, TRIAGE_TILE, device=DEVICE)
    out_p = OUT/'triage_cnn.onnx'
    torch.onnx.export(m, dummy, str(out_p), opset_version=CFG['onnx_opset'],
                      input_names=['sar_tile'], output_names=['triage_logits'],
                      dynamic_axes={'sar_tile': {0:'batch'}, 'triage_logits': {0:'batch'}})
    print(f'✓ Triage ONNX: {out_p.name}  ({out_p.stat().st_size/1e6:.1f} MB)')

export_unet_onnx()
export_triage_onnx()

In [ ]:
# ─── CELL 14: SAVE METRICS & FINAL SUMMARY ──────────────────────────────────
from datetime import datetime
import json

metrics = {
    'training_completed_utc': datetime.utcnow().isoformat(),
    'hardware': {'gpu_count': N_GPU, 'gpu_name': torch.cuda.get_device_name(0),
                 'cuda': torch.version.cuda},
    'config': CFG,
    'segmentation': {'best_val_iou': round(best_seg_iou, 4), 'history': seg_history},
    'triage': {'best_val_acc': round(best_tri_acc, 4), 'label_map': TRIAGE_LABELS,
               'history': tri_history},
    'paths': {'oil_img': str(OIL_IMG_DIR), 'oil_mask': str(OIL_MASK_DIR),
              'lookalike': str(LIKE_IMG_DIR), 'no_oil': str(NOIL_IMG_DIR)},
}
with open(OUT/'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print('─── Final Summary ──────────────────────────────────────────────────')
print(f'  U-Net  Best Val IoU  : {best_seg_iou:.4f}')
print(f'  Triage Best Val Acc  : {best_tri_acc:.4f}')
print('\n─── Artifacts ──────────────────────────────────────────────────────')
for p in sorted(OUT.glob('*')):
    if p.is_file():
        print(f'  {p.name:40s} {p.stat().st_size/1e6:7.2f} MB')
print('\nDone. Pull weights + ONNX + metrics.json to your local repo.')